# ECE-GY 9143 Lab 4 Part A: Distributed Deep Learning (NYU HPC)

This notebook orchestrates Part A experiments by calling `lab4_ddp.py` from shell cells.

## Before running
- Request an interactive job with up to 4 GPUs (same GPU type).
- Update the environment activation command in each `%%bash` cell.
- Keep `num_workers=2` and default Lab 2 hyper-parameters.
- Use a persistent dataset path (example used below: `/scratch/$USER/hpml_lab4_data`).

## What this notebook covers
- Q1 to Q4.1 experiment runs and measurement collection.
- Q4.2, Q5, Q6 answer prompts (report writing sections).


## Q1: Computational Efficiency vs Batch Size (Single GPU)

Run 2 epochs per batch size and report the **2nd epoch `train_time`** from `Q1 RESULT`.
Start with 32 and increase by 4x until OOM.


In [ ]:
%%bash
# TODO: replace with your own environment activation
source /scratch/ht2667/miniconda3/bin/activate hpml_lab

DATA_PATH=${DATA_PATH:-/scratch/$USER/hpml_lab4_data}

# Required sweep starts here
python lab4_ddp.py --run q1 --batch_size 32   --data_path "$DATA_PATH"
python lab4_ddp.py --run q1 --batch_size 128  --data_path "$DATA_PATH"
python lab4_ddp.py --run q1 --batch_size 512  --data_path "$DATA_PATH"
python lab4_ddp.py --run q1 --batch_size 2048 --data_path "$DATA_PATH"

# Continue 4x until OOM if your GPU can hold more
# python lab4_ddp.py --run q1 --batch_size 8192 --data_path "$DATA_PATH"


## Q2: Speedup Measurement (2 GPU / 4 GPU)

For each batch size used in Q1 (e.g., 32/128/512), run 2 epochs and report the **2nd epoch `total_time`**.
Then compute speedup using `Speedup = T(1-GPU total_time) / T(N-GPU total_time)` for each setup.

> This is weak scaling because batch size per GPU is fixed while effective batch size increases with GPU count.


In [ ]:
%%bash
# TODO: replace with your own environment activation
source /scratch/ht2667/miniconda3/bin/activate hpml_lab

DATA_PATH=${DATA_PATH:-/scratch/$USER/hpml_lab4_data}
BATCHES=(32 128 512)

for bs in "${BATCHES[@]}"; do
  echo "=== Q2 | 2 GPUs | batch_size_per_gpu=${bs} ==="
  torchrun --nproc_per_node=2 lab4_ddp.py --run q2 --batch_size "$bs" --data_path "$DATA_PATH"

  echo "=== Q2 | 4 GPUs | batch_size_per_gpu=${bs} ==="
  torchrun --nproc_per_node=4 lab4_ddp.py --run q2 --batch_size "$bs" --data_path "$DATA_PATH"
done


## Q3: Computation vs Communication + Bandwidth Utilization

Run 2-GPU and 4-GPU for each batch size. Report the 2nd epoch `train_time` from Q3.

For each `(num_gpu, batch_size_per_gpu)`:
- `compute_time = Q1 train_time`
- `comm_time = Q3 DDP train_time - Q1 train_time`

For Q3.2 bandwidth utilization, use:
- `model_size_bytes = num_params * 4`
- `allreduce_volume = 2 * (N-1)/N * model_size_bytes`
- `effective_bandwidth = allreduce_volume / comm_time_per_iter`


In [ ]:
%%bash
# TODO: replace with your own environment activation
source /scratch/ht2667/miniconda3/bin/activate hpml_lab

DATA_PATH=${DATA_PATH:-/scratch/$USER/hpml_lab4_data}
BATCHES=(32 128 512)

for bs in "${BATCHES[@]}"; do
  echo "=== Q3 | 2 GPUs | batch_size_per_gpu=${bs} ==="
  torchrun --nproc_per_node=2 lab4_ddp.py --run q3 --batch_size "$bs" --data_path "$DATA_PATH"

  echo "=== Q3 | 4 GPUs | batch_size_per_gpu=${bs} ==="
  torchrun --nproc_per_node=4 lab4_ddp.py --run q3 --batch_size "$bs" --data_path "$DATA_PATH"
done


## Q4.1: Large Batch Training (4 GPUs, 5 Epochs)

Use the largest successful `batch_size_per_gpu` from Q1 (`MAX_BS`).
Report 5th epoch average loss and accuracy, then compare with Lab 2 (`batch_size=128`, single GPU).


In [ ]:
%%bash
# TODO: replace with your own environment activation
source /scratch/ht2667/miniconda3/bin/activate hpml_lab

DATA_PATH=${DATA_PATH:-/scratch/$USER/hpml_lab4_data}
MAX_BS=512   # TODO: replace with the largest successful batch size from your Q1 run

torchrun --nproc_per_node=4 lab4_ddp.py --run q4 --batch_size "$MAX_BS" --epochs 5 --data_path "$DATA_PATH"


## Q4.2: How to Improve Training Accuracy for Large Batch (Report Section)

Read reference [4] and provide two remedies in your report.
Suggested points to cover:
- Remedy 1
- Remedy 2


## Q5: What Is Passed on Network? (Report Section)

Answer whether gradients are the only communicated messages across learners, and explain.


## Q6: What If We Only Communicate Gradients? (Report Section)

For the `batch_size_per_gpu=512`, 4-GPU setup, explain whether communicating only gradients is sufficient, with reasoning based on [4].
